In [1]:
import pandas as pd
import numpy as np

In [2]:
df_fi = pd.read_parquet("dane/interim/fact_inka_records_2023_2026.parquet")

FileNotFoundError: [Errno 2] No such file or directory: 'dane/interim/fact_inka_records_2023_2026.parquet'

In [7]:
# ============================================================
# HARD WYKLUCZENIA
# Kategorie które nie pochodzą z magazynu centralnego SPAR
# lub nie są towarami fizycznymi
# ============================================================

# Prasa i gazety (osobny system zamawiania - kolportaż)
asid_prasa = {132, 133, 273}

# Kiosk - bilety, zapalniczki, leki, baterie (nie z magazynu)
asid_kiosk = {245, 277, 278, 279, 283, 284, 285, 288, 290, 293, 275}

# Doładowania telefoniczne i startery (usługi, nie towary)
asid_doladowania = {47, 48, 255, 290, 330, 331, 332, 334, 335, 337, 338}

# Opakowania i techniczne (nie towary handlowe)
asid_techniczne = {
    256,  # OPAKOWANIA ZWROTNE
    352,  # reklamówki
    247,  # OPAKOWANIA /przemysłowe
    254,  # xx KOSZTY / RABATY
    244,  # Domyślny DO WERYFIKACJI
    354,  # xinne
}

# Łączna lista wykluczeń
lista_hard_wykluczen = (
    asid_prasa |
    asid_kiosk |
    asid_doladowania |
    asid_techniczne
)

print(f"Łącznie do twardego wykluczenia: {len(lista_hard_wykluczen)} AsId")

df_fi_hard = df_fi.copy()

# Filtrowanie AsId
df_fi_hard = df_fi_hard[~df_fi_hard['AsId'].isin(lista_hard_wykluczen)]

# Filtrowanie dokumentów bez NrDok
df_fi_hard = df_fi_hard[df_fi_hard['NrDok'].notna()]

print(f"Przed: {len(df_fi):,} rekordów")
print(f"Po:    {len(df_fi_hard):,} rekordów")
print(f"Usunięto: {len(df_fi) - len(df_fi_hard):,} rekordów")

Łącznie do twardego wykluczenia: 30 AsId
Przed: 4,260,543 rekordów
Po:    4,106,901 rekordów
Usunięto: 153,642 rekordów


--- 
# Koniec filtrowania "HARD"
---

# Filtry "SOFT"

## Soft filtrowanie — po TowId/SKU

### Kryterium 1: PctDniZerowych per TowId
- wyrzucamy TowId z PctDniZerowych > próg
- próg do ustalenia po policzeniu rozkładu

### Kryterium 2: Towary ważone per TowId  
- wyrzucamy TowId z ułamkowymi IloscPlus
- z wyjątkami (pieczywo - połówki)

---

# Metryki

### Główne:

* Ilosc_sprzedazy      — ile jednostek sprzedano (to optymalizujemy)
* Ilosc_transakcji     — liczba transakcji sprzedaży
* Dni_ze_sprzedaza     — ile dni miało sprzedaż
* Pct_dni_zerowych     — % dni bez sprzedaży
* Avg_dzienna_ilosc    — średnia dzienna ilość sprzedaży
* Ilosc_SKU            — liczba produktów w kategorii
---
### Pomocnicze — do cash flow:
* Wartosc_sprzedazy         — łączne obroty (oszacowanie cash flow)
* Avg_wartosc_per_SKU       — średnia wartość per produkt
* Ilosc_zamowien            — liczba PZ (przyjęć z magazynu)
* Avg_wartosc_zamowienia    — średnia wartość jednego zamówienia PZ
---
### Pozostałe:
- KategoriaABC (A/B/C)

### Per SKU (decyzja operacyjna — później):
- te same metryki per TowId
- KategoriaABC per SKU

### Cash flow / Working Capital:
- COGS miesięczny (~367k zł)
- Working Capital (~92k zł przy tygodniowym cyklu)
- potwierdzenie częstotliwości zamówień z PZ

---

# Terminy branżowe / domenowe

### Cash flow zakupowy:
- OTB (Open-To-Buy) — budżet dostępny na zakupy w danym okresie
- COGS (Cost of Goods Sold) — koszt sprzedanych towarów

### Zamrożony kapitał:
- Working Capital — kapitał obrotowy zamrożony w zapasie
- Inventory Value — wartość zapasu
- DOS (Days of Supply) — ile dni zapasu mamy na stanie
- DIO (Days Inventory Outstanding) — ile dni towar leży w magazynie zanim zostanie sprzedany

### Dead Stock:
- Obsolete Inventory — przestarzały/przeterminowany zapas
- Slow Movers — towary z niską rotacją
- Dead Stock — towary bez ruchu
- Shrinkage — straty (kradzieże, uszkodzenia, przeterminowanie)

### Rotacja:
- Inventory Turnover — wskaźnik rotacji zapasów (obroty / średni zapas)
- Sell-Through Rate — % sprzedanego towaru z dostarczonego

### Kategoryzacja:
- ABC Analysis — podział na kategorie A/B/C wg wartości/rotacji
- Pareto (80/20) — 20% SKU generuje 80% obrotów

---
# PL vs EN - polskie "odpowiedniki".
---
### Używane zamiennie PL/EN:
- DOS / Pokrycie zapasu (dni)
- Working Capital / Kapitał obrotowy
- Sell-Through Rate / Wskaźnik sprzedaży

### Częściej po polsku:
- Rotacja zapasów (nie Inventory Turnover)
- Zapas / Stan magazynowy (nie Inventory)
- Zamówienie (nie Order)

### Powszechnie znane po angielsku (bez polskich wersji):
- OTB, COGS, ABC Analysis
- Dead Stock, Slow Movers
- Inventory Turnover
- KPI, Dashboard

---

In [10]:
df_fi_hard[['AsId', 'NazwaAsort', 'NazwaAsortCleanName']].value_counts().reset_index()

,AsId,NazwaAsort,NazwaAsortCleanName,0
0,326,MARKA WŁASNA SPAR,marka wlasna spar,161711
1,322,PAPIEROSY,papierosy,130653
2,86,JOGURTY /nabiał,jogurty nabial,130109
3,115,WARZYWA,warzywa,121788
4,102,WODY,wody,121439
...,...,...,...,...
264,249,EKO /nabiał,eko nabial,5
265,248,SMAKOSZ /dodatki spożywcze,smakosz dodatki spozywcze,5
266,136,CHLEBY I POCHODNE /dietetyczne,chleby i pochodne dietetyczne,5
267,183,MATERIAŁY PALNE /przemysłowe,materialy palne przemyslowe,5


In [21]:
df_fi_hard_21 = df_fi_hard[df_fi_hard['TypDok']==21]

---
# sprzedaz zerowa  
---

In [103]:
# Pełny kalendarz dni (zakres dat w danych)

df_fi_calendar = df_fi_hard_21.copy() 

all_days = pd.date_range(
    start= df_fi_calendar['Data'].min(),
    end= df_fi_calendar['Data'].max(),
    freq='D'
)
total_days = len(all_days)

# Dni ze sprzedażą per AsId
df_ze_sprzedaza = (
    df_fi_calendar[df_fi_calendar['IloscPlus'] > 0]
    .groupby(['AsId', 'NazwaAsort'])['Data']
    .nunique()
    .reset_index()
    .rename(columns={'Data': 'DniZeSprzedaza'})
)

# Procent dni zerowych
df_ze_sprzedaza['DniZeSprzedazaPct'] = (
    df_ze_sprzedaza['DniZeSprzedaza'] / total_days * 100
).round(2)

df_ze_sprzedaza['DniZerowych'] = (
    total_days - df_ze_sprzedaza['DniZeSprzedaza']
).round(2)

print(f"Łączna liczba dni w kalendarzu: {total_days}")
# dni_sale = df_ze_sprzedaza.sort_values('DniZerowych', ascending=False)

Łączna liczba dni w kalendarzu: 1126


In [104]:
df_ze_sprzedaza

,AsId,NazwaAsort,DniZeSprzedaza,DniZeSprzedazaPct,DniZerowych
0,2,3 ALKOHOLE INNE POWYŻEJ 18%,362,32.15,764
1,3,1 PIWO BUTELKA,952,84.55,174
2,4,1 PIWO PUSZKA,952,84.55,174
3,5,2 WINA MUSUJĄCE I SZAMPANY do 18%,779,69.18,347
4,6,3 WHISKY RUM BRANDY RUM BOURBON,906,80.46,220
...,...,...,...,...,...
234,347,MUSY,703,62.43,423
235,348,MIĘSO PACZKOWANE,20,1.78,1106
236,350,WARZYWA MROŻONE,949,84.28,177
237,351,GRZYBY MROŻONE,192,17.05,934


In [113]:
# Dni kiedy sklep był otwarty
dni_otwarte = set(df_fi_calendar['Data'].unique())
total_dni_otwarte = len(dni_otwarte)

print(f"Łączna liczba dni w kalendarzu: {total_days}")
print(f"Dni kiedy sklep był otwarty: {total_dni_otwarte}")
print(f"Dni zamknięcia: {total_days - total_dni_otwarte}")

Łączna liczba dni w kalendarzu: 1126
Dni kiedy sklep był otwarty: 952
Dni zamknięcia: 174


In [17]:
df_ze_sprzedaza['pct_dni_zerowych_skorygowany'] = (
    (total_dni_otwarte - df_ze_sprzedaza['DniZeSprzedaza']) / 
    total_dni_otwarte * 100
).round(2)

# Rozkład per przedział
bins = [0, 35, 65, 100]
labels = ['regularna (<35%)', 'szara strefa (35-65%)', 'słaba (>65%)']
df_ze_sprzedaza['kategoria'] = pd.cut(
    df_ze_sprzedaza['pct_dni_zerowych_skorygowany'], 
    bins=bins, labels=labels, include_lowest=True
)
print(df_ze_sprzedaza['kategoria'].value_counts())

regularna (<35%)         153
słaba (>65%)              71
szara strefa (35-65%)     34
Name: kategoria, dtype: int64


In [19]:
# Listy AsId per kategoria zerowej sprzedaży
lista_regularna = set(
    df_ze_sprzedaza[df_ze_sprzedaza['kategoria'] == 'regularna (<35%)']['AsId']
)

lista_szara = set(
    df_ze_sprzedaza[df_ze_sprzedaza['kategoria'] == 'szara strefa (35-65%)']['AsId']
)

lista_slaba = set(
    df_ze_sprzedaza[df_ze_sprzedaza['kategoria'] == 'słaba (>65%)']['AsId']
)

print(f"Regularna (<45% zer):      {len(lista_regularna)} AsId")
print(f"Szara strefa (45-75% zer): {len(lista_szara)} AsId")
print(f"Słaba (>75% zer):          {len(lista_slaba)} AsId")

Regularna (<45% zer):      153 AsId
Szara strefa (45-75% zer): 34 AsId
Słaba (>75% zer):          71 AsId


In [ ]:
df_fi[df_fi['TowId'].isin(lista_regularna)][['TowId', 'NazwaTow', 'AsId']].drop_duplicates()

,TowId,NazwaTow,AsId
1198,74,ZAPALNICZKA DZIEWCZYNY A50,191
7653,92,BUŁKA ZIARNO MIX,119
44366,82,BULKA HOT DOG Z SEREM,119
48177,95,BUŁKA KAJZERKA PRECELEK,119
55235,86,Bulka poznanska 0.1,119
73110,75,PIECZYWO MIX,119
97416,100,PĄCZEK PISTACJA SZT,119
98101,81,BULKA Z SEREM MAREL,119
98255,93,BAJGLE SEZAM/ MAK 70G MAREL,119
100340,78,"MIKOŁAJ, CHOINKA 100G MAREL",117


In [24]:
print(sorted(lista_regularna))

[3, 4, 5, 6, 7, 8, 10, 11, 13, 14, 20, 22, 23, 25, 26, 28, 29, 30, 31, 32, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 46, 58, 66, 70, 73, 74, 75, 77, 78, 79, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 97, 98, 99, 100, 101, 102, 104, 105, 106, 108, 112, 113, 115, 117, 118, 119, 120, 121, 122, 123, 124, 127, 128, 132, 147, 148, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 168, 185, 191, 194, 195, 196, 197, 198, 200, 202, 204, 205, 206, 207, 208, 211, 213, 216, 217, 219, 221, 222, 223, 224, 226, 228, 230, 232, 233, 235, 236, 237, 238, 240, 242, 243, 245, 256, 257, 273, 306, 307, 313, 314, 319, 322, 323, 326, 341, 343, 344, 345, 347, 350, 352, 355]


In [25]:
len(lista_regularna)+len(lista_slaba)+len(lista_szara)

258

In [26]:
# Lista AsId do wyrzucenia - ważone
lista_wykluczen_wazone = {80, 215, 114, 68}

# Lista AsId do wyrzucenia - inne (kioski, doładowania, opakowania, "x")
lista_wykluczen_inne = {256, 190, 9, 169, 354, 283, 255, 337, 338, 334, 335, 284, 277}

# Łącznie
lista_wykluczen = lista_wykluczen_wazone | lista_wykluczen_inne
print(f"Łącznie do wyrzucenia: {len(lista_wykluczen)} AsId")

Łącznie do wyrzucenia: 17 AsId


In [ ]:
lista_do_wziecia = lista_wykluczen

In [ ]:
df_fi[df_fi['AsId'].isin(lista_do_wziecia)][['AsId', 'NazwaAsort']].drop_duplicates().sort_values('AsId')

,AsId,NazwaAsort
455736,3,1 PIWO BUTELKA
411655,4,1 PIWO PUSZKA
1190920,8,3 WÓDKI I NAP.ALK POWYŻEJ 18%
3254708,10,CHUSTECZKI HIGIENICZNE /papierniczo hig
1276237,25,BUDYNIE /dodatki deserowe
...,...,...
248107,326,MARKA WŁASNA SPAR
3248435,344,DODATKI
1371690,345,ZUPKI CHIŃSKIE
4027114,350,WARZYWA MROŻONE


In [ ]:
# Czy AsId ma jakiekolwiek transakcje z ułamkową ilością
wazone = (
    df_fi_calendar[df_fi_calendar['AsId'].isin(lista_do_wziecia)]
    .groupby('AsId')
    .apply(lambda x: (x['IloscPlus'] % 1 != 0).any())
    .reset_index()
    .rename(columns={0: 'CzyWazone'})
)

lista_wazone = set(wazone[wazone['CzyWazone'] == True]['AsId'])
print(f"Ważonych AsId: {len(lista_wazone)}")
print(sorted(lista_wazone))

Ważonych AsId: 9
[70, 112, 115, 117, 120, 211, 219, 221, 257]


In [ ]:
df_fi[df_fi['AsId'].isin(lista_wazone)][['AsId', 'NazwaAsort']].drop_duplicates().sort_values('AsId')

,AsId,NazwaAsort
1452559,70,WĘDLINY PACZKOWANE
2252093,112,OWOCE
2360929,115,WARZYWA
98599,117,PIECZYWO
156790,120,CHLEBY
1976606,211,PRZETWORY RYBNE /ryby
928010,219,CIASTKA
2756464,221,CUKIERKI
199070,257,PIEROGI KOPYTKA KROKIETY INNE


In [39]:
lista_wazone = lista_wazone - {70, 117, 120, 211, 219}

In [ ]:
print(sorted(lista_wazone))

[112, 115, 221, 257]


In [41]:
# lista_do_wziecia = lista_do_wziecia - lista_wazone
# print(sorted(lista_do_wziecia))
# print(len(lista_do_wziecia))

In [ ]:
df_fi[df_fi['AsId'].isin({132,352, 256, 190, 9, 169, 354, 283, 255, 337, 338, 334, 335, 284, 277})][['AsId', 'NazwaAsort']].drop_duplicates().sort_values('AsId')

,AsId,NazwaAsort
4241504,9,x PAPIERNICZO HIGIENICZNE
2484561,132,PRASA
4235108,169,x PRZEMYSŁOWE
4241155,190,ZABAWKI /przemysłowe
4217461,255,Doładowania PayUp - bez terminala
4008653,256,OPAKOWANIA ZWROTNE
4214289,277,05 ZAPALNICZKI KIOSK
4235787,283,06 BILETY W KIOSKU
4241085,284,11 LEKI KIOSK
4236337,334,HEYAH


In [43]:
lista_do_wziecia = lista_do_wziecia - {132, 352}

In [ ]:
df_fi[df_fi['AsId'].isin(lista_do_wziecia)][['AsId', 'NazwaAsort']].drop_duplicates().sort_values('AsId')

,AsId,Nazwa_asort
456071,3,1 PIWO BUTELKA
411979,4,1 PIWO PUSZKA
1191414,8,3 WÓDKI I NAP.ALK POWYŻEJ 18%
3255783,10,CHUSTECZKI HIGIENICZNE /papierniczo hig
1276774,25,BUDYNIE /dodatki deserowe
...,...,...
3508116,322,PAPIEROSY
248308,326,MARKA WŁASNA SPAR
3249509,344,DODATKI
1372241,345,ZUPKI CHIŃSKIE


In [ ]:
df_fi[df_fi['AsId'].isin(lista_wazone)][['AsId', 'NazwaAsort']].drop_duplicates().sort_values('AsId')

,AsId,NazwaAsort
2252093,112,OWOCE
2360929,115,WARZYWA
2756464,221,CUKIERKI
199070,257,PIEROGI KOPYTKA KROKIETY INNE


In [ ]:
# Czy AsId ma jakiekolwiek transakcje z ułamkową ilością
wazone_end = (
    df_fi_hard_21[df_fi_hard_21['AsId'].isin(lista_do_wziecia)]
    .groupby('AsId')
    .apply(lambda x: (x['IloscPlus'] % 1 != 0).any())
    .reset_index()
    .rename(columns={0: 'CzyWazone'})
)

lista_wazone_end = set(wazone_end[wazone_end['CzyWazone'] == True]['AsId'])
print(f"Ważonych AsId: {len(lista_wazone_end)}")
print(sorted(lista_wazone_end))

Ważonych AsId: 9
[70, 112, 115, 117, 120, 211, 219, 221, 257]


In [ ]:
# Znajdź ważone TowId (ułamkowe ilości)
wazone_tow = (
    df_fi_hard_21[df_fi_hard_21['AsId'].isin(lista_do_wziecia)]
    .groupby('TowId')
    .apply(lambda x: (x['IloscPlus'] % 1 != 0).any())
    .reset_index()
    .rename(columns={0: 'CzyWazone'})
)

lista_wazone_tow = set(wazone_tow[wazone_tow['CzyWazone'] == True]['TowId'])
print(f"Ważonych TowId do wyrzucenia: {len(lista_wazone_tow)}")

Ważonych TowId do wyrzucenia: 85


In [ ]:
df_fi.shape

(4260543, 40)

In [ ]:
# Transakcje z całkowitą ilością w "ważonych" AsId
nie_wazone_w_wazonej = (
    df_fi_hard_21[df_fi_hard_21['AsId'].isin(lista_wazone)]
    [lambda x: x['IloscPlus'] % 1 == 0]
    .groupby(['AsId', 'Nazwa', 'TowId'])
    ['IloscPlus'].sum()
    .reset_index()
    .sort_values('AsId')
)

nie_wazone_w_wazonej

,AsId,Nazwa,TowId,IloscPlus
0,70,BALERON COPPA 100G BALCERZAK,66903,10.0
122,70,PARÓWKI SOKOLIKI 140G SOKOŁÓW,14574,1906.0
123,70,PARÓWKI WIEPRZOWE 200G GOODVALLEY,78334,6.0
124,70,PARÓWKI Z FILETA Z KURCZAKA 180G TARCZYŃSKI,39188,521.0
125,70,PARÓWKI Z INDYKA BERLINKI 250G MORLINY,50579,141.0
...,...,...,...,...
1065,257,KLUSKI ŚLĄSKIE 400G GRZEŚKOWIAK,54589,1470.0
1064,257,KLUSKI Z MAKIEM 500G KUCHNIA POLKI,67437,5.0
1063,257,KLUSKI SZARE WĘDZONKA 280G SPAR,66071,350.0
1075,257,KROKIETY MOZARELLA I PIECZARKI 450G KUCHNIA POLKI,49521,16.0


In [ ]:
Asid_Do_Wziecia = {257, 3, 4, 8, 10, 25, 28, 29, 31, 32, 34, 37, 40, 41, 42, 43, 44, 46, 313, 322, 326, 70, 74, 75, 83, 84, 85, 86, 87, 88, 345, 90, 344, 89, 93, 94, 95, 350, 97, 98, 99, 100, 101, 102, 104, 106, 112, 115, 117, 118, 119, 120, 121, 124, 128, 151, 153, 154, 155, 156, 159, 160, 161, 164, 165, 166, 168, 196, 198, 202, 204, 205, 207, 211, 216, 217, 219, 221, 223, 228, 230, 232, 233, 235, 236, 237, 240, 245}
df_fi_selected_asid = df_fi[df_fi['AsId'].isin(Asid_Do_Wziecia)]
df_fi_selected_asid.columns

Index(['DokId', 'Kolejnosc', 'NrPozycji', 'TowId', 'TypPoz', 'IloscPlus',
       'IloscMinus', 'CenaPrzedRab', 'CenaPoRab', 'Wartosc', 'CenaDet', 'Data',
       'KolejnyWDniu', 'NrDok', 'TypDok', 'AktywnyDok', 'Razem', 'DoZaplaty',
       'Zaplacono', 'AsId', 'JMId', 'NazwaTow', 'EAN', 'Opis1', 'Producent',
       'Marza', 'Stawka', 'AktywnyTow', 'CleanName', 'NazwaAsort',
       'PrefiksDok', 'Dokument', 'Typ_pozycji', 'Wplyw_na_stan',
       'Kolumna_ilosciowa', 'Mnoznik', 'Typ_ruchu', 'Czy_niechciane',
       'ilosc_bazowa', 'ilosc_netto'],
      dtype='object')

In [ ]:
df_fi_selected_asid[['TypDok', 'PrefiksDok', 'TypPoz']].value_counts().reset_index().sort_values(by='TypDok')

,TypDok,PrefiksDok,TypPoz,0
1,2,PZ,1,219704
16,8,ZWPAR,1,231
17,9,PW,1,51
6,10,RW,4,12045
5,14,BO,3,19497
4,16,REM,3,28533
3,18,PRZEC,6,68704
14,19,ZAMR_PRZEC,0,2816
0,21,DF,4,2945791
7,23,ST,4,11293


In [63]:
df_fi_21 = df_fi.copy()[df_fi['TypDok']==21]

In [65]:
# Szacunek cash flow
wartosc_total = df_fi_hard_21['Wartosc'].sum()
miesiace = 37

wartosc_miesiecznie = wartosc_total / miesiace
marza = 0.25  # środek zakresu 20-30%
koszt_zakupu_miesiecznie = wartosc_miesiecznie * (1 - marza)
zamrozony_kapital = koszt_zakupu_miesiecznie  # 70-80% środek

print(f"Obroty łącznie:              {wartosc_total:,.0f} zł")
print(f"Obroty miesięcznie:          {wartosc_miesiecznie:,.0f} zł")
print(f"Koszt zakupu miesięcznie:    {koszt_zakupu_miesiecznie:,.0f} zł")
print(f"Szacowany zamrożony kapitał: {zamrozony_kapital:,.0f} zł/miesiąc")

Obroty łącznie:              18,093,082 zł
Obroty miesięcznie:          489,002 zł
Koszt zakupu miesięcznie:    366,752 zł
Szacowany zamrożony kapitał: 366,752 zł/miesiąc


In [64]:
# Szacunek cash flow
wartosc_total = df_fi_21['Wartosc'].sum()
miesiace = 37

wartosc_miesiecznie = wartosc_total / miesiace
marza = 0.25  # środek zakresu 20-30%
koszt_zakupu_miesiecznie = wartosc_miesiecznie * (1 - marza)
zamrozony_kapital = koszt_zakupu_miesiecznie * 0.75  # 70-80% środek

print(f"Obroty łącznie:              {wartosc_total:,.0f} zł")
print(f"Obroty miesięcznie:          {wartosc_miesiecznie:,.0f} zł")
print(f"Koszt zakupu miesięcznie:    {koszt_zakupu_miesiecznie:,.0f} zł")
print(f"Szacowany zamrożony kapitał: {zamrozony_kapital:,.0f} zł/miesiąc")

Obroty łącznie:              18,366,491 zł
Obroty miesięcznie:          496,392 zł
Koszt zakupu miesięcznie:    372,294 zł
Szacowany zamrożony kapitał: 279,220 zł/miesiąc
